In [1]:
import json
import yaml
import pandas as pd
from matplotlib import pyplot as plt

In [2]:
df_yes = pd.read_csv("Yes.csv")
df_discussion = pd.read_csv("Discussion.csv")

In [3]:
def fix_formatting(s):
    tokens = [t.strip() for t in s.lower().split(",")]
    tokens.sort()
    return tokens

df_yes["tokens"] = df_yes["Classification Keywords"].apply(lambda s: fix_formatting(s))
df_yes.head()

,Classification Keywords,"Keep (Yes, No, Discussion)",Title,Consider (Y/N/T),Unnamed: 4,Unnamed: 2,Cites,Authors,Year,Source,...,g_coverage,star_count,year_first,year_last,acc1,acc2,acc5,acc20,hA,tokens
0,"Pruning, Robust, Performance",Yes,""" Understanding Robustness Lottery"": A Compara...",T,1,0,0.0,"Z Li, S Liu, X Yu, K Bhavya, J Cao, DJ Daniel…",2022.0,arXiv preprint arXiv …,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[performance, pruning, robust]"
1,"Deep Learning, Robust, Learning-Enabled System...",Yes,"""know What You Know"": Predicting Behavior for ...",T,12,111,2.0,M.A. Langford,2021.0,Proceedings - 2021 International Symposium on ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[deep learning, learning-enabled system, robus..."
2,"Adversarial, Robust",Yes,(Certified!!) Adversarial Robustness for Free!,T,17,9,0.0,"N Carlini, F Tramer, JZ Kolter",2022.0,arXiv preprint arXiv:2206.10550,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[adversarial, robust]"
3,"Neural Network, Uncertainty",Yes,A -Distribution Based Operator for Enhancing O...,T,171,117,1.0,"N Antonello, PN Garner",2020.0,IEEE Signal Processing Letters,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[neural network, uncertainty]"
4,"Adversarial, Spiking Neural Network, Robust, R...",Yes,A Comprehensive Analysis on Adversarial Robust...,T,313,168,17.0,"Saima Sharmin, P. Panda, Syed Shakib Sarwar, C...",2019.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[adversarial, resilience, robust, spiking neur..."


In [4]:
def tokens_to_dict(df, col):
    kw_dict = {}
    for _, row in df.iterrows():
        for t in row[col]:
            if t not in kw_dict:
                kw_dict[t] = 0
            kw_dict[t] +=1

    # kw_dict = {k: v for k, v in sorted(kw_dict.items(), key = lambda item: item[1])}
    kw_dict = {k: v for k, v in sorted(kw_dict.items(), key = lambda item: item[1], reverse = True)}
    
    return kw_dict

In [5]:
kw_dict = tokens_to_dict(df_yes, "tokens")
with open("kw_dict.json", "w") as f:
    json.dump(kw_dict, f, indent = 4)

In [6]:
with open("kw_changes.yml", "r") as f:
    kw_rules = yaml.load(f, Loader=yaml.FullLoader)

def fix_keywords(tokens, kw_rules):
    for k in kw_rules.keys():
        dest = kw_rules[k]["dest"]
        source = kw_rules[k]["source"]
        if dest == "to_delete":
            pass
        else:
            for source_kw in source:
                for t in tokens:
                    if source_kw == t:
                        tokens.remove(t)
                        for d in dest:
                            tokens.append(d)
    tokens = list(set(tokens))
    tokens.sort()
    return tokens

df_yes["tokens_agg"] = df_yes["tokens"].apply(lambda s: fix_keywords(s, kw_rules))

kw_dict_fix = tokens_to_dict(df_yes, "tokens_agg")
with open("kw_dict_fix.json", "w") as f:
    json.dump(kw_dict_fix, f, indent = 4)

In [7]:
df_yes.head()

,Classification Keywords,"Keep (Yes, No, Discussion)",Title,Consider (Y/N/T),Unnamed: 4,Unnamed: 2,Cites,Authors,Year,Source,...,star_count,year_first,year_last,acc1,acc2,acc5,acc20,hA,tokens,tokens_agg
0,"Pruning, Robust, Performance",Yes,""" Understanding Robustness Lottery"": A Compara...",T,1,0,0.0,"Z Li, S Liu, X Yu, K Bhavya, J Cao, DJ Daniel…",2022.0,arXiv preprint arXiv …,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[performance, pruning, robust]","[performance, pruning, robust]"
1,"Deep Learning, Robust, Learning-Enabled System...",Yes,"""know What You Know"": Predicting Behavior for ...",T,12,111,2.0,M.A. Langford,2021.0,Proceedings - 2021 International Symposium on ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[deep learning, learning-enabled system, robus...","[deep learning, learning-enabled system, robus..."
2,"Adversarial, Robust",Yes,(Certified!!) Adversarial Robustness for Free!,T,17,9,0.0,"N Carlini, F Tramer, JZ Kolter",2022.0,arXiv preprint arXiv:2206.10550,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[adversarial, robust]","[adversarial, robust]"
3,"Neural Network, Uncertainty",Yes,A -Distribution Based Operator for Enhancing O...,T,171,117,1.0,"N Antonello, PN Garner",2020.0,IEEE Signal Processing Letters,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[neural network, uncertainty]","[neural network, uncertainty]"
4,"Adversarial, Spiking Neural Network, Robust, R...",Yes,A Comprehensive Analysis on Adversarial Robust...,T,313,168,17.0,"Saima Sharmin, P. Panda, Syed Shakib Sarwar, C...",2019.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[adversarial, resilience, robust, spiking neur...","[adversarial, resilience, robust, spiking neur..."


In [8]:
kw_low_freq = [k for k in kw_dict_fix if (kw_dict_fix[k] < 2)]
len(kw_low_freq)

113

In [12]:
def check_delete(tokens, kw_low_freq):
    for t in tokens:
        if t in kw_low_freq or t == "to_delete":
            return True
    return False

df_yes["to_delete"] = df_yes["tokens_agg"].apply(lambda s: check_delete(s, kw_low_freq))

print("This many would be ignored: {}".format(len(df_yes[df_yes["to_delete"] == True])))
print("This many would be kept: {}".format(len(df_yes) - len(df_yes[df_yes["to_delete"] == True])))

This many would be ignored: 141
This many would be kept: 562


In [24]:
df_keep = df_yes[df_yes["to_delete"] == False]
len(df_keep)

562

In [25]:
df_keep.sort_values(by="GSRank", inplace=True)
df_keep = df_keep.drop(["to_delete", "tokens", "Unnamed: 2", "Unnamed: 4"], axis=1)
df_keep.head()

/var/folders/7d/nc4thym14h33bjbwklwp_9zw0000gn/T/ipykernel_76194/499853368.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_keep.sort_values(by="GSRank", inplace=True)


,Classification Keywords,"Keep (Yes, No, Discussion)",Title,Consider (Y/N/T),Cites,Authors,Year,Source,Publisher,ArticleURL,...,g_coverage,star_count,year_first,year_last,acc1,acc2,acc5,acc20,hA,tokens_agg
466,"neural networks, adversarial robustness",Yes,Towards Evaluating the Robustness of Neural Ne...,T,2534.0,N. Carlini,2017.0,Proceedings - IEEE Symposium on Security and P...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[adversarial robustness, neural networks]"
280,"Computer Vision, Certified Robustness, Perturb...",Yes,Learning perturbation sets for robust machine ...,T,38.0,"E Wong, JZ Kolter",2020.0,arXiv preprint arXiv:2007.08450,arxiv.org,https://arxiv.org/abs/2007.08450,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[computer vision, perturbations, robust]"
631,"Certified Robustness, Smoothing, Neural Network",Yes,Input-Specific Robustness Certification for Ra...,Y,1.0,"Ruoxin Chen, Jie Li, Junchi Yan, Ping Li, Bin ...",2021.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[neural network, robust, smoothing]"
122,"Classifier, Robust, Benchmark",Yes,Benchmarking Neural Network Robustness to Comm...,T,1185.0,"Dan Hendrycks, Thomas G. Dietterich",2018.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[benchmark, classifier, robust]"
313,"Certified Robustness, Adversarial Attacks, Neu...",Yes,Neural Network Robustness Verification on GPUs,T,13.0,"Christoph Müller, Gagandeep Singh, Markus Püsc...",2020.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[adversarial attacks, neural network, robust]"


In [26]:
df_keep.to_csv("ordered_articles.csv")

In [29]:
def remove_kw(tokens, kw_low_freq):
    new_tokens = tokens
    for t in tokens:
        if t in kw_low_freq or t == "to_delete":
            new_tokens.remove(t)
    return new_tokens

def check_delete_2(tokens):
    if len(tokens) > 0:
        return False
    else:
        return True

df_yes["tokens_agg_2"] = df_yes["tokens_agg"].apply(lambda s: remove_kw(s, kw_low_freq))
df_yes["to_delete_2"] = df_yes["tokens_agg_2"].apply(lambda s: check_delete_2(s))

print("This many would be ignored: {}".format(len(df_yes[df_yes["to_delete_2"] == True])))
print("This many would be kept: {}".format(len(df_yes) - len(df_yes[df_yes["to_delete_2"] == True])))

This many would be ignored: 4
This many would be kept: 699
